# Notebook 04 — Construcción de Features: Infección Postoperatoria
## TFG — Predicción de Complicaciones Postquirúrgicas · Ingeniería en Ciencia de Datos

**Objetivo**: construir la matriz de features `X` (1 fila por paciente) y unirla con la variable objetivo `y` de infección postoperatoria para entrenar modelos de predicción.

**Fuentes de datos**:

| Segmento | Archivos | Features extraídas |
|---|---|---|
| `static/` | `000_POBLACION_DIANA`, `006_CHARLSON_NUEVO` | Edad, sexo, comorbilidades |
| `intra/` | `009_INTERVENCIONES` | Tipo anestesia, ASA, duración cirugía, tipo procedimiento, urgencia |
| `pre/` | `007_SOLICITUD`, `013_CONSTANTES*`, `010_LABORATORIO*` | Tipo ingreso, constantes vitales preop, analíticas preop |

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import sys
from pathlib import Path

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

DIV     = PROJECT_ROOT / "data" / "data_divided"
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print(f"Proyecto : {PROJECT_ROOT}")
print(f"Dividido : {DIV}")
print(f"Resultados: {RESULTS}")


Proyecto : /Users/ikerarias/Desktop/TFG
Dividido : /Users/ikerarias/Desktop/TFG/data/data_divided
Resultados: /Users/ikerarias/Desktop/TFG/results


## 1. Variable objetivo `y`

Cargamos el CSV `y_infeccion_postop.csv` generado a partir de `016_HOSPITALIZACION_DIAGNOSTICO_PRINCIPAL.csv` (diagnósticos ICD-10 de infección postoperatoria). Contiene 391 positivos (3.18%) sobre 12,288 pacientes.

In [15]:
y = pd.read_csv(RESULTS / "y_infeccion_postop.csv")
y["id_paciente"] = y["id_paciente"].astype(str).str.strip()

print(f"Variable objetivo cargada: {y.shape}")
print(y["infeccion_postop"].value_counts().rename({0: "No infección (0)", 1: "Infección (1)"}))
print(f"\nPrevalencia: {y['infeccion_postop'].mean()*100:.2f}%")


Variable objetivo cargada: (13662, 3)
infeccion_postop
No infección (0)    13249
Infección (1)         413
Name: count, dtype: int64

Prevalencia: 3.02%


## 2. Features demográficas (static/000)

Edad en el momento de la cirugía, sexo y tipo de residencia.

In [16]:
# Cargar referencia quirúrgica para obtener fecha de cirugía (para calcular edad)
df_ref = pd.read_csv(DIV / "intra" / "009_INTERVENCIONES_QUIRURGICAS_INTERVENCION.csv",
                     engine="python", on_bad_lines="skip",
                     usecols=["id_paciente", "fecha_evento"])
df_ref.columns = df_ref.columns.str.strip()
df_ref["id_paciente"] = df_ref["id_paciente"].astype(str).str.strip()
df_ref["fecha_cirugia"] = pd.to_datetime(df_ref["fecha_evento"], errors="coerce")
df_ref = df_ref.drop_duplicates(subset="id_paciente")[["id_paciente", "fecha_cirugia"]]

# Cargar demografía (separador ;)
df_demo = pd.read_csv(DIV / "static" / "000_POBLACION_DIANA.csv",
                      sep=";", engine="python", on_bad_lines="skip")
df_demo.columns = df_demo.columns.str.strip()
df_demo["id_paciente"] = df_demo["Id_Paciente"].astype(str).str.strip()
df_demo["fecnac"] = pd.to_datetime(df_demo["fecnac"], dayfirst=True, errors="coerce")

# Merge para calcular edad en el momento de la cirugía
df_demo = df_demo.merge(df_ref, on="id_paciente", how="left")
df_demo["edad"] = ((df_demo["fecha_cirugia"] - df_demo["fecnac"]).dt.days / 365.25).round(1)

# Codificación de sexo
df_demo["sexo_mujer"] = (df_demo["sexo"].str.strip() == "Mujer").astype(int)

feat_demo = df_demo[["id_paciente", "edad", "sexo_mujer"]].copy()

print(f"Features demográficas: {feat_demo.shape}")
print(f"  edad   — nulos: {feat_demo['edad'].isna().sum()}  |  media: {feat_demo['edad'].mean():.1f} años")
print(f"  sexo_mujer — nulos: {feat_demo['sexo_mujer'].isna().sum()}  |  % mujeres: {feat_demo['sexo_mujer'].mean()*100:.1f}%")
feat_demo.head(4)


Features demográficas: (13662, 3)
  edad   — nulos: 1374  |  media: 62.0 años
  sexo_mujer — nulos: 0  |  % mujeres: 48.0%


,id_paciente,edad,sexo_mujer
0,17598,70.2,0
1,84611,34.2,1
2,39314,74.4,1
3,60484,83.3,0


## 3. Comorbilidades de Charlson (static/006_CHARLSON_NUEVO)

Índice de Charlson estándar + condiciones adicionales (HTA, dislipemia, arritmia, asma…).

In [17]:
df_ch = pd.read_csv(DIV / "static" / "006_COMORBILIDADES_CHARLSON_NUEVO.csv",
                    sep=";", engine="python", on_bad_lines="skip")
df_ch.columns = df_ch.columns.str.strip()
df_ch["id_paciente"] = df_ch["id_paciente"].astype(str).str.strip()

# Columnas de comorbilidades (todas las que no son id ni años)
cols_ch = [c for c in df_ch.columns if c not in ("id_paciente",)]
# Convertir a numérico y calcular score Charlson total (suma de charlson_*)
cols_charlson = [c for c in cols_ch if c.startswith("charlson_")]
cols_extra    = [c for c in cols_ch if not c.startswith("charlson_")]

for col in cols_ch:
    df_ch[col] = pd.to_numeric(df_ch[col], errors="coerce").fillna(0).astype(int)

df_ch["charlson_score"] = df_ch[cols_charlson].sum(axis=1)

# Mantener columnas clínicas más relevantes + score total
# Agrupar por id_paciente (puede haber duplicados por fecha)
cols_keep = ["id_paciente", "charlson_score"] + cols_extra
feat_ch = df_ch[cols_keep].groupby("id_paciente").max().reset_index()

print(f"Features Charlson: {feat_ch.shape}")
print(f"  charlson_score — media: {feat_ch['charlson_score'].mean():.2f}  |  max: {feat_ch['charlson_score'].max()}")
print(f"  Condiciones extra: {cols_extra}")
feat_ch.head(3)


Features Charlson: (17403, 16)
  charlson_score — media: 1.73  |  max: 16
  Condiciones extra: ['EII', 'angina', 'arritmia', 'hta', 'dislipemia', 'linfoma', 'leucemia', 'coagulopatia', 'sangrado_gastro', 'asma', 'bronquiectasia', 'fibrosis_quis', 'enf_pulmon_intersticial', 'inf_bronquial_cronica']


,id_paciente,charlson_score,EII,angina,arritmia,hta,dislipemia,linfoma,leucemia,coagulopatia,sangrado_gastro,asma,bronquiectasia,fibrosis_quis,enf_pulmon_intersticial,inf_bronquial_cronica
0,1000,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,10009,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0
2,1001,1,1,0,0,1,1,0,0,0,0,0,0,0,0,0


## 4. Features quirúrgicas (intra/009)

Características del acto quirúrgico: tipo de anestesia, ASA, duración, urgencia, tipo de procedimiento.

In [18]:
df_qx = pd.read_csv(DIV / "intra" / "009_INTERVENCIONES_QUIRURGICAS_INTERVENCION.csv",
                    engine="python", on_bad_lines="skip")
df_qx.columns = df_qx.columns.str.strip()
df_qx["id_paciente"] = df_qx["id_paciente"].astype(str).str.strip()

# Duración como numérico
df_qx["duracion_min"] = pd.to_numeric(df_qx["Duración Intervención"], errors="coerce")

# ASA como ordinal (I→1, II→2, etc.)
asa_map = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6}
df_qx["asa"] = df_qx["Riesgo anestésico (ASA)"].astype(str).str.strip().map(asa_map)

# Variables categóricas → dummies
df_qx["urgente"] = (df_qx["Prioridad de la intervención"].astype(str).str.lower()
                    .isin(["urgente","urgente preferente"])).astype(int)
df_qx["cirugia_mayor"] = (df_qx["Tipo de procedimiento quirúrgico"].astype(str).str.upper()
                           == "CIRUGIA MAYOR").astype(int)
df_qx["infeccion_intraop"] = (df_qx["Infección intraoperatoria"].astype(str).str.upper()
                               == "SI").astype(int)

# Tipo de anestesia → dummies (las más frecuentes)
df_qx["anestesia_general"] = df_qx["Tipo anestesia"].astype(str).str.upper().str.contains("GENERAL").astype(int)
df_qx["anestesia_locoregional"] = df_qx["Tipo anestesia"].astype(str).str.upper().str.contains("INTRADURAL|EPIDURAL|PLEXO|LOCAL|BLOQUEO").astype(int)

# 1 fila por paciente (primer episodio principal ya seleccionado en la separación)
feat_qx = df_qx[["id_paciente", "duracion_min", "asa", "urgente", "cirugia_mayor",
                  "infeccion_intraop", "anestesia_general", "anestesia_locoregional"]
                ].drop_duplicates(subset="id_paciente")

print(f"Features quirúrgicas: {feat_qx.shape}")
print(f"  duracion_min — media: {feat_qx['duracion_min'].mean():.1f} min  |  nulos: {feat_qx['duracion_min'].isna().sum()}")
print(f"  asa          — nulos: {feat_qx['asa'].isna().sum()}")
print(f"  urgentes:    {feat_qx['urgente'].sum()}")
print(f"  cirugía mayor: {feat_qx['cirugia_mayor'].sum()}")
print(f"  infección intraop: {feat_qx['infeccion_intraop'].sum()}")
feat_qx.head(4)


Features quirúrgicas: (12288, 8)
  duracion_min — media: 76.9 min  |  nulos: 0
  asa          — nulos: 0
  urgentes:    958
  cirugía mayor: 9353
  infección intraop: 0


,id_paciente,duracion_min,asa,urgente,cirugia_mayor,infeccion_intraop,anestesia_general,anestesia_locoregional
0,46243,140,3,0,1,0,0,1
1,36960,45,1,0,1,0,0,1
2,35808,75,1,0,0,0,1,0
3,22369,290,3,0,1,0,0,1


## 5. Features de laboratorio preoperatorio (pre/010_*)

Concatenamos todos los archivos `010_LABORATORIO*.csv` del directorio `pre/`, pivotamos de formato largo a ancho y tomamos el **último valor** de cada analítica antes de la cirugía.


In [19]:
lab_files = sorted((DIV / "pre").glob("010_LABORATORIO*.csv"))
print(f"Archivos laboratorio encontrados: {len(lab_files)}")
for f in lab_files:
    print(f"  {f.name}")

labs = pd.concat(
    [pd.read_csv(f, engine="python", on_bad_lines="skip") for f in lab_files],
    ignore_index=True
)
labs.columns = labs.columns.str.strip()
labs["id_paciente"] = labs["id_paciente"].astype(str).str.strip()
labs["fecha_evento"] = pd.to_datetime(labs["fecha_evento"], errors="coerce")
labs["Resultado_num"] = pd.to_numeric(
    labs["Resultado texto"].astype(str).str.replace(",", "."), errors="coerce"
)
print(f"\nLabs totales: {labs.shape[0]:,} filas  |  {labs['id_paciente'].nunique():,} pacientes")
print("\nTests más frecuentes:")
print(labs["Desc prueba(Resultado)"].value_counts().head(20).to_string())


Archivos laboratorio encontrados: 4
  010_LABORATORIO.csv
  010_LABORATORIO_2019_1.csv
  010_LABORATORIO_2019_2.csv
  010_LABORATORIO_2019_3.csv

Labs totales: 3,925 filas  |  87 pacientes

Tests más frecuentes:
Desc prueba(Resultado)
San-Linfocitos                                           154
San-Monocitos                                            154
San-Neutrófilos                                          154
San-Basófilos                                            153
San-Eosinófilos                                          153
Srm-Índice de hemólisis                                   82
San-Hematíes                                              79
San-Hemoglobina corpuscular media eritrocitaria (HCM)     79
San-Hemoglobina                                           79
San-Leucocitos                                            79
San-Concentración de hemoglobina corpuscular media        78
San-Hematocrito                                           78
San-Plaquetas                    

In [20]:
# Tests de relevancia clínica para infección
TESTS_INTERES = [
    "San-Hemoglobina", "San-Leucocitos", "San-Plaquetas",
    "San-Neutrófilos", "San-Linfocitos",
    "Srm-Creatinina", "Srm-Glucosa", "Srm-Sodio", "Srm-Potasio", "Srm-Urea",
    "Srm-Proteina C reactiva (PCR)",
    "Pla-Tiempo (Indice) de protrombina (%)",
    "San-Hematocrito",
]
labs_fil = labs[labs["Desc prueba(Resultado)"].isin(TESTS_INTERES)].copy()
print(f"Registros tras filtrado: {labs_fil.shape[0]:,}")
print(labs_fil["Desc prueba(Resultado)"].value_counts().to_string())

# Último valor preoperatorio por paciente y test
feat_lab = (
    labs_fil.sort_values("fecha_evento")
    .groupby(["id_paciente", "Desc prueba(Resultado)"])["Resultado_num"]
    .last()
    .unstack(level=1)
)

# Nombres de columna limpios
def clean_col(c):
    return ("lab_" + c.split("-", 1)[-1]
            .strip()
            .lower()
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
            .replace("%", "pct")
            .replace("á", "a").replace("é", "e").replace("í", "i")
            .replace("ó", "o").replace("ú", "u").replace("ó", "o"))

feat_lab.columns = [clean_col(c) for c in feat_lab.columns]
feat_lab = feat_lab.reset_index()

print(f"\nFeatures laboratorio: {feat_lab.shape}")
print(feat_lab.columns.tolist())
feat_lab.head(3)


Registros tras filtrado: 1,069
Desc prueba(Resultado)
San-Linfocitos                            154
San-Neutrófilos                           154
San-Hemoglobina                            79
San-Leucocitos                             79
San-Hematocrito                            78
San-Plaquetas                              78
Srm-Creatinina                             78
Srm-Glucosa                                78
Srm-Potasio                                71
Srm-Sodio                                  70
Srm-Urea                                   63
Pla-Tiempo (Indice) de protrombina (%)     51
Srm-Proteina C reactiva (PCR)              36

Features laboratorio: (58, 14)
['id_paciente', 'lab_tiempo_indice_de_protrombina_pct', 'lab_hematocrito', 'lab_hemoglobina', 'lab_leucocitos', 'lab_linfocitos', 'lab_neutrofilos', 'lab_plaquetas', 'lab_creatinina', 'lab_glucosa', 'lab_potasio', 'lab_proteina_c_reactiva_pcr', 'lab_sodio', 'lab_urea']


,id_paciente,lab_tiempo_indice_de_protrombina_pct,lab_hematocrito,lab_hemoglobina,lab_leucocitos,lab_linfocitos,lab_neutrofilos,lab_plaquetas,lab_creatinina,lab_glucosa,lab_potasio,lab_proteina_c_reactiva_pcr,lab_sodio,lab_urea
0,10583,69.0,37.4,12.6,10.49,13.4,7.5528,248.0,0.69,91.2,4.06,70.97,143.0,36.0
1,1070,27.0,39.7,13.1,6.20,32.4,3.4100,171.0,0.90,116.8,4.10,NaN,144.0,39.0
2,11821,100.0,44.0,14.4,15.33,29.3,63.4000,281.0,0.74,128.7,4.80,6.65,142.0,37.0


## 6. Constantes vitales preoperatorias (pre/013_*)

Concatenamos los archivos `013_CONSTANTES_*.csv` (formato largo), convertimos valores y tomamos la **última medición** preoperatoria de cada ítem por paciente.


In [21]:
cte_files = sorted((DIV / "pre").glob("013_CONSTANTES*.csv"))
print(f"Archivos constantes encontrados: {len(cte_files)}")
for f in cte_files:
    print(f"  {f.name}")

ctes = pd.concat(
    [pd.read_csv(f, engine="python", on_bad_lines="skip") for f in cte_files],
    ignore_index=True
)
ctes.columns = ctes.columns.str.strip()
ctes["id_paciente"] = ctes["id_paciente"].astype(str).str.strip()
ctes["fecha_evento"] = pd.to_datetime(ctes["fecha_evento"], errors="coerce")
ctes["Valor_num"] = pd.to_numeric(
    ctes["Valor (form cte)"].astype(str).str.replace(",", "."), errors="coerce"
)
print(f"\nConstantes totales: {ctes.shape[0]:,} filas")
print("\nÍtems disponibles:")
print(ctes["Item (form cte)"].value_counts().to_string())


Archivos constantes encontrados: 3
  013_CONSTANTES_1.csv
  013_CONSTANTES_2.csv
  013_CONSTANTES_3.csv

Constantes totales: 360,661 filas

Ítems disponibles:
Item (form cte)
Presión arterial (SIST)                        75273
Presión arterial (DIAS)                        75247
Frecuencia Cardiaca (lpm)                      71177
Temperatura ºC                                 52107
Saturación capilar de O2 (%) pulsioximetría    35200
Peso                                           20825
talla (cm)                                     12496
IMC                                            12460
Estado Neurológico                              3918
Frecuencia Respiratoria (rpm)                   1958


In [22]:
# Ítems de interés clínico
ITEMS_INTERES = {
    "Presión arterial (SIST)": "ta_sist",
    "Presión arterial (DIAS)": "ta_dias",
    "Frecuencia Cardiaca (lpm)": "fc",
    "Temperatura (ºc)": "temperatura",
    "Saturación O2": "sato2",
    "Glucemia Basal": "glucemia_basal",
    "Peso (kg)": "peso_kg",
    "Talla (cm)": "talla_cm",
    "Índice de masa corporal": "imc",
}

# Filtrar y mapear nombres
ctes_fil = ctes[ctes["Item (form cte)"].isin(ITEMS_INTERES)].copy()
ctes_fil["item_clean"] = ctes_fil["Item (form cte)"].map(ITEMS_INTERES)
print(f"Registros tras filtrado: {ctes_fil.shape[0]:,}")
print(ctes_fil["item_clean"].value_counts().to_string())

# Calcular IMC si falta: peso / (talla/100)^2
feat_vital = (
    ctes_fil.sort_values("fecha_evento")
    .groupby(["id_paciente", "item_clean"])["Valor_num"]
    .last()
    .unstack(level=1)
).reset_index()
feat_vital.columns.name = None
feat_vital = feat_vital.rename(columns={c: f"vital_{c}" if c != "id_paciente" else c
                                         for c in feat_vital.columns})

# Calcular IMC si no está disponible directamente
if "vital_peso_kg" in feat_vital.columns and "vital_talla_cm" in feat_vital.columns:
    mask = feat_vital.get("vital_imc", pd.Series(dtype=float)).isna()
    if mask.any():
        talla_m = feat_vital["vital_talla_cm"] / 100
        imc_calc = feat_vital["vital_peso_kg"] / (talla_m ** 2)
        if "vital_imc" not in feat_vital.columns:
            feat_vital["vital_imc"] = imc_calc
        else:
            feat_vital.loc[mask, "vital_imc"] = imc_calc[mask]

print(f"\nFeatures constantes: {feat_vital.shape}")
print(feat_vital.columns.tolist())
feat_vital.head(3)


Registros tras filtrado: 221,697
item_clean
ta_sist    75273
ta_dias    75247
fc         71177

Features constantes: (11501, 4)
['id_paciente', 'vital_fc', 'vital_ta_dias', 'vital_ta_sist']


,id_paciente,vital_fc,vital_ta_dias,vital_ta_sist
0,10007,69.0,73.0,124.0
1,10009,69.0,95.0,166.0
2,1001,80.0,81.0,131.0


## 7. Features de solicitud y contacto anestesia (pre/007, pre/008)

`007_INTERVENCIONES_QUIRURGICAS_SOLICITUD.csv` contiene datos de la solicitud quirúrgica preoperatoria (tipo ingreso, ASA solicitud, proceso quirúrgico, prioridad).  
`008_INTERVENCIONES_CONTACTO_ANESTESIA.csv` tiene el ASA registrado por el anestesista en la consulta preanestesia.


In [23]:
df_sol = pd.read_csv(DIV / "pre" / "007_INTERVENCIONES_QUIRURGICAS_SOLICITUD.csv",
                     engine="python", on_bad_lines="skip")
df_sol.columns = df_sol.columns.str.strip()
df_sol["id_paciente"] = df_sol["id_paciente"].astype(str).str.strip()

print("Columnas 007:", df_sol.columns.tolist())
print(f"Filas: {len(df_sol):,}  |  Pacientes únicos: {df_sol['id_paciente'].nunique():,}")

# ASA solicitud (mismo mapa que intra)
asa_map = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6}
asa_col_sol = [c for c in df_sol.columns if "asa" in c.lower() or "riesgo" in c.lower()]
print("Columnas ASA en solicitud:", asa_col_sol)

if asa_col_sol:
    df_sol["asa_solicitud"] = df_sol[asa_col_sol[0]].astype(str).str.strip().map(asa_map)

# Ingreso urgente
ingreso_col = [c for c in df_sol.columns if "ingreso" in c.lower()]
print("Columnas ingreso:", ingreso_col)
if ingreso_col:
    df_sol["ingreso_urgente"] = (df_sol[ingreso_col[0]].astype(str).str.upper()
                                  .str.contains("URGENTE|EMERGEN")).astype(int)

# Suspensión fármacos
farm_col = [c for c in df_sol.columns if "farm" in c.lower() or "suspen" in c.lower()]
print("Columnas fármacos:", farm_col)
if farm_col:
    df_sol["suspension_farmacos"] = (df_sol[farm_col[0]].astype(str).str.upper()
                                      .isin(["SI", "SÍ", "1", "TRUE"])).astype(int)

cols_sol = ["id_paciente"]
for c in ["asa_solicitud", "ingreso_urgente", "suspension_farmacos"]:
    if c in df_sol.columns:
        cols_sol.append(c)

feat_sol = df_sol[cols_sol].drop_duplicates(subset="id_paciente")
print(f"\nFeatures solicitud: {feat_sol.shape}")
feat_sol.head(3)


Columnas 007: ['cod Solicitud', 'id_paciente', 'Episodio SOLIC', 'id_episodio', 'fecha_evento', 'cod Organizacion servicio SOL', 'cod Centro SOL', 'cod Servicio SOL', 'cod Seccion SOL', 'cod Estado solicitud', 'Estado solicitud', 'cod Subestado solicitud', 'Subestado solicitud', 'Tipo ingreso', 'Tipo anestesia', 'Riesgo anestesico (ASA)', 'Tipo de procedimiento', 'Descrip Kit material', 'Requiere autotransfusion', 'Lateralidad', 'Suspension farmacos', 'Prioridad de la Intervencion', 'Tipo lista', 'cod Diagnostico', 'Proceso quirurgico', 'Autoconcertada', 'Hora (prevista solicitud)', 'cod Bloque quirurgico', 'cod Equipo quirurgico solicitado', 'cod Punto atencion (solicitud)', 'Fecha Preoperatorio (contacto anestesista)', 'Codigo gizabide (solicitante)']
Filas: 12,101  |  Pacientes únicos: 12,101
Columnas ASA en solicitud: ['Riesgo anestesico (ASA)']
Columnas ingreso: ['Tipo ingreso']
Columnas fármacos: ['Suspension farmacos']

Features solicitud: (12101, 4)


,id_paciente,asa_solicitud,ingreso_urgente,suspension_farmacos
0,46243,3.0,0,1
1,36960,1.0,0,0
2,35808,1.0,0,0


## 8. Merge final → matriz X

Unimos todos los bloques de features (demográficas, Charlson, quirúrgicas, laboratorio, constantes vitales, solicitud) mediante `left join` sobre `id_paciente` anclado en la variable objetivo `y`.


In [24]:
feature_blocks = {
    "demograficas": feat_demo,
    "charlson":     feat_ch,
    "quirurgicas":  feat_qx,
    "laboratorio":  feat_lab,
    "constantes":   feat_vital,
    "solicitud":    feat_sol,
}

X = y[["id_paciente"]].copy()
for name, df_feat in feature_blocks.items():
    before = len(X)
    X = X.merge(df_feat, on="id_paciente", how="left")
    print(f"  + {name:15s} → {df_feat.shape[1]-1:3d} features "
          f"| filas: {len(X):,} (antes {before:,})")

print(f"\nX final shape: {X.shape}")
print(f"Pacientes sin ninguna feature: {(X.drop(columns='id_paciente').isna().all(axis=1)).sum()}")
X.head(3)


  + demograficas    →   2 features | filas: 13,662 (antes 13,662)
  + charlson        →  15 features | filas: 13,662 (antes 13,662)
  + quirurgicas     →   7 features | filas: 13,662 (antes 13,662)
  + laboratorio     →  13 features | filas: 13,662 (antes 13,662)
  + constantes      →   3 features | filas: 13,662 (antes 13,662)
  + solicitud       →   3 features | filas: 13,662 (antes 13,662)

X final shape: (13662, 44)
Pacientes sin ninguna feature: 0


,id_paciente,edad,sexo_mujer,charlson_score,EII,angina,arritmia,hta,dislipemia,linfoma,...,lab_potasio,lab_proteina_c_reactiva_pcr,lab_sodio,lab_urea,vital_fc,vital_ta_dias,vital_ta_sist,asa_solicitud,ingreso_urgente,suspension_farmacos
0,17598,70.2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,84.0,80.0,117.0,3.0,0.0,1.0
1,84611,34.2,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,91.0,84.0,101.0,1.0,0.0,0.0
2,39314,74.4,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,80.0,80.0,153.0,NaN,0.0,0.0


## 9. Análisis de missings, imputación y exportación

Evaluamos el porcentaje de valores faltantes por feature, imputamos la mediana para numéricas y eliminamos las columnas con >80% de nulos (sin información útil).


In [25]:
X_feat = X.drop(columns="id_paciente")

# Missings
miss_pct = X_feat.isna().mean().sort_values(ascending=False)
print("Porcentaje de nulos por feature (top 30):")
print((miss_pct.head(30) * 100).round(1).to_string())

# Eliminar features con > 80% nulos
cols_drop = miss_pct[miss_pct > 0.80].index.tolist()
print(f"\nFeatures eliminadas por >80% nulos ({len(cols_drop)}): {cols_drop}")
X_feat = X_feat.drop(columns=cols_drop)

# Imputación por mediana
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(
    imputer.fit_transform(X_feat),
    columns=X_feat.columns,
    index=X_feat.index
)

print(f"\nX tras imputación: {X_imp.shape}")
print(f"Nulos restantes: {X_imp.isna().sum().sum()}")

# Resumen estadístico rápido
print("\nEstadísticas básicas (primeras 10 cols):")
print(X_imp.iloc[:, :10].describe().round(2).to_string())


Porcentaje de nulos por feature (top 30):
lab_proteina_c_reactiva_pcr             99.8
lab_tiempo_indice_de_protrombina_pct    99.7
lab_urea                                99.7
lab_potasio                             99.7
lab_sodio                               99.7
lab_creatinina                          99.6
lab_neutrofilos                         99.6
lab_glucosa                             99.6
lab_linfocitos                          99.6
lab_plaquetas                           99.6
lab_hematocrito                         99.6
lab_leucocitos                          99.6
lab_hemoglobina                         99.6
hta                                     75.9
dislipemia                              75.9
charlson_score                          75.9
EII                                     75.9
angina                                  75.9
inf_bronquial_cronica                   75.9
arritmia                                75.9
enf_pulmon_intersticial                 75.9
fibrosis_quis

In [26]:
# Exportar X y y para los notebooks de modelado
X_export = pd.concat([X[["id_paciente"]], X_imp], axis=1)
X_export.to_csv(RESULTS / "X_infeccion.csv", index=False)
print(f"X exportada → results/X_infeccion.csv  |  shape: {X_export.shape}")

# y ya existe pero la confirmamos
print(f"y ya exportada → results/y_infeccion_postop.csv  |  shape: {y.shape}")
print(f"\nDistribución y:")
print(y["infeccion_postop"].value_counts().to_string())
print(f"  Ratio clase mayoritaria / minoritaria: {y['infeccion_postop'].value_counts()[0] / y['infeccion_postop'].value_counts()[1]:.1f}:1")
print(f"\nCategorías de infección:")
print(y["categoria_infeccion"].value_counts().to_string())


X exportada → results/X_infeccion.csv  |  shape: (13662, 31)
y ya exportada → results/y_infeccion_postop.csv  |  shape: (13662, 3)

Distribución y:
infeccion_postop
0    13249
1      413
  Ratio clase mayoritaria / minoritaria: 32.1:1

Categorías de infección:
categoria_infeccion
ninguna    13249
resp         193
sepsis        67
itu           53
ssi           40
cateter       35
other         25
